In [12]:
import pandas as pd
import numpy as np

In [12]:
#Ques 1: Group users by signup month, compute monthly retention rates, and output a cohort matrix showing % of each cohort active in each subsequent month.
users = pd.DataFrame({ "user_id": range(1,21), "signup_date": pd.date_range("2024-01-01",periods=20,freq="3D") }) 
activity = pd.DataFrame({ "user_id": [1,1,1,2,2,3,3,3,4,4,5,5,6,7,8,9,10,11,12,13], "activity_date": pd.to_datetime([ "2024-01-01","2024-02-01","2024-03-01", "2024-01-04","2024-02-04","2024-01-07", "2024-02-07","2024-03-07","2024-01-10", "2024-02-10","2024-01-13","2024-03-13", "2024-01-16","2024-01-19","2024-01-22", "2024-01-25","2024-01-28","2024-01-31", "2024-02-03","2024-02-06"]) }) 
act = activity.merge(users, on="user_id")

act = activity.merge(users, on="user_id") 
act["cohort"] = act["signup_date"].dt.to_period("M") 
act["activity_month"] = act["activity_date"].dt.to_period("M") 
act["cohort_age"] = (act["activity_month"]-act["cohort"]).apply(lambda x: x.n)
display(act)

matrix = (act.groupby(['cohort', 'cohort_age'])['user_id'].nunique().reset_index()).pivot_table(index='cohort', columns='cohort_age', values='user_id')
display(matrix)

retention = matrix.divide(matrix[0], axis=0)*100

display(retention.round(1))

,user_id,activity_date,signup_date,cohort,activity_month,cohort_age
0,1,2024-01-01,2024-01-01,2024-01,2024-01,0
1,1,2024-02-01,2024-01-01,2024-01,2024-02,1
2,1,2024-03-01,2024-01-01,2024-01,2024-03,2
3,2,2024-01-04,2024-01-04,2024-01,2024-01,0
4,2,2024-02-04,2024-01-04,2024-01,2024-02,1
5,3,2024-01-07,2024-01-07,2024-01,2024-01,0
6,3,2024-02-07,2024-01-07,2024-01,2024-02,1
7,3,2024-03-07,2024-01-07,2024-01,2024-03,2
8,4,2024-01-10,2024-01-10,2024-01,2024-01,0
9,4,2024-02-10,2024-01-10,2024-01,2024-02,1


cohort_age,0,1,2
cohort,,,
2024-01,11.0,4.0,3.0
2024-02,2.0,NaN,NaN


cohort_age,0,1,2
cohort,,,
2024-01,100.0,36.4,27.3
2024-02,100.0,NaN,NaN


In [19]:
#Ques 2: Split clickstream events into sessions using a 30-minute gap rule. Compute session metrics and overall conversion rate.

cs = pd.DataFrame({ "user_id": [1,1,1,1,2,2,2,3,3,3,3], 
                   "event_time": pd.to_datetime([ "2024-01-01 10:00","2024-01-01 10:05","2024-01-01 10:45", "2024-01-01 10:50","2024-01-01 11:00","2024-01-01 11:10", "2024-01-01 12:00","2024-01-01 14:00","2024-01-01 14:05", "2024-01-01 14:10","2024-01-01 14:15"]), 
                   "event_type": ["view"]*10 + ["purchase"] }).sort_values(["user_id","event_time"])

cs['time_diff'] = cs.groupby('user_id')['event_time'].diff()

cs["new_session"] = ( (cs["time_diff"] > pd.Timedelta(minutes=30)) | cs["time_diff"].isnull() ).astype(int)

cs["session_id"] = cs.groupby("user_id")["new_session"].cumsum()

display(cs)

sessions = cs.groupby(["user_id","session_id"]).agg( 
                                                    start_time = ("event_time","min"), 
                                                    end_time = ("event_time","max"), 
                                                    page_count = ("event_time","count"), 
                                                    has_conversion = ("event_type", lambda x: "purchase" in x.values) 
                                                ).reset_index()

sessions["duration_min"] = ( (sessions["end_time"]-sessions["start_time"]).dt.total_seconds()/60 )

display(sessions)

print(f"Conversion rate: {sessions.has_conversion.mean()*100:.1f}%")

,user_id,event_time,event_type,time_diff,new_session,session_id
0,1,2024-01-01 10:00:00,view,NaT,1,1
1,1,2024-01-01 10:05:00,view,0 days 00:05:00,0,1
2,1,2024-01-01 10:45:00,view,0 days 00:40:00,1,2
3,1,2024-01-01 10:50:00,view,0 days 00:05:00,0,2
4,2,2024-01-01 11:00:00,view,NaT,1,1
5,2,2024-01-01 11:10:00,view,0 days 00:10:00,0,1
6,2,2024-01-01 12:00:00,view,0 days 00:50:00,1,2
7,3,2024-01-01 14:00:00,view,NaT,1,1
8,3,2024-01-01 14:05:00,view,0 days 00:05:00,0,1
9,3,2024-01-01 14:10:00,view,0 days 00:05:00,0,1


,user_id,session_id,start_time,end_time,page_count,has_conversion,duration_min
0,1,1,2024-01-01 10:00:00,2024-01-01 10:05:00,2,False,5.0
1,1,2,2024-01-01 10:45:00,2024-01-01 10:50:00,2,False,5.0
2,2,1,2024-01-01 11:00:00,2024-01-01 11:10:00,2,False,10.0
3,2,2,2024-01-01 12:00:00,2024-01-01 12:00:00,1,False,0.0
4,3,1,2024-01-01 14:00:00,2024-01-01 14:15:00,4,True,15.0


Conversion rate: 20.0%


In [11]:
#Ques 3: Calculate Recency, Frequency, Monetary scores, assign quintiles (1-5), and segment customers.

analysis_date = pd.Timestamp("2024-03-01") 
tx = pd.DataFrame({ "customer_id": [1,1,1,2,2,3,3,3,3,4,5,5,6,6,6,6], "transaction_date": pd.to_datetime([ "2024-02-28","2024-02-15","2024-01-10", "2024-02-25","2024-01-05", "2024-02-20","2024-02-10","2024-01-20","2024-01-05", "2024-01-15","2024-02-27","2024-02-01", "2024-02-26","2024-02-15","2024-01-25","2024-01-10"]), "amount": [150,200,100,300,250,180,220,190,160,80,400,350,200,180,190,210] })

display(tx)

rfm = tx.groupby('customer_id').agg(
    recency = ('transaction_date', lambda x: (analysis_date - x.max()).days),
    frequency = ('customer_id', 'count'),
    monetary = ('amount', 'sum')
).reset_index()

rfm['R'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1], duplicates='drop')
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5], duplicates='drop')
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5], duplicates='drop')

rfm['rfm score'] = rfm[['R', 'F', 'M']].astype(int).sum(axis=1)

rfm['weighted score'] = (rfm['R'].astype(int) * 0.5 +
                         rfm['F'].astype(int) * 0.3 +
                         rfm['M'].astype(int) * 0.2)

def segment(row):
    s,r = row["rfm score"], int(row['R'])
    if s>=9: return 'Champion'
    elif r>=4 and s>=6: return "Loyal"
    elif r>=3 and s>=5: return "Potential"
    elif r<=2: return "At Risk"
    else: return "Needs Attention"

def weighted_segment(row):
    s = row["weighted score"]
    if s>4: return 'Champion'
    elif s>3: return "Loyal"
    elif s>2: return "Potential"
    elif s>1: return "At Risk"
    else: return "Needs Attention"

rfm["segment"] = rfm.apply(segment, axis = 1)
rfm["weighter segment"] = rfm.apply(weighted_segment, axis=1)

display(rfm)

,customer_id,transaction_date,amount
0,1,2024-02-28,150
1,1,2024-02-15,200
2,1,2024-01-10,100
3,2,2024-02-25,300
4,2,2024-01-05,250
5,3,2024-02-20,180
6,3,2024-02-10,220
7,3,2024-01-20,190
8,3,2024-01-05,160
9,4,2024-01-15,80


,customer_id,recency,frequency,monetary,R,F,M,rfm score,weighted score,segment,weighter segment
0,1,2,3,450,5,3,1,9,3.6,Champion,Loyal
1,2,5,2,550,3,1,2,6,2.2,Potential,Potential
2,3,10,4,750,2,4,3,9,2.8,Champion,Potential
3,4,46,1,80,1,1,1,3,1.0,At Risk,Needs Attention
4,5,3,2,750,5,2,4,11,3.9,Champion,Loyal
5,6,4,4,780,4,5,5,14,4.5,Champion,Champion


In [17]:
#Ques 4: Flag values more than 2 standard deviations from the 30-day centred rolling mean.

np.random.seed(42) 
dates = pd.date_range("2024-01-01", periods=100, freq="D") 
values = np.random.normal(100, 10, 100) 
# Inject anomalies 
values[20] = 150 # spike 
values[45] = 50 # drop 
values[70] = 160 # spike 
df_ts = pd.DataFrame({"date":dates, "value":values})

df_ts['rolling mean'] = df_ts['value'].rolling(window=30, center=True).mean()
df_ts['rolling_std'] = df_ts['value'].rolling(window=30, center=True).std()

df_ts['z-score'] = (df_ts['value'] - df_ts['rolling mean']) / df_ts['rolling_std']

df_ts['anomaly'] = abs(df_ts['z-score']) > 2

display(df_ts)

anomalies = df_ts[df_ts['anomaly']]

display(anomalies)

,date,value,rolling mean,rolling_std,z-score,anomaly
0,2024-01-01,104.967142,NaN,NaN,NaN,False
1,2024-01-02,98.617357,NaN,NaN,NaN,False
2,2024-01-03,106.476885,NaN,NaN,NaN,False
3,2024-01-04,115.230299,NaN,NaN,NaN,False
4,2024-01-05,97.658466,NaN,NaN,NaN,False
...,...,...,...,...,...,...
95,2024-04-05,85.364851,NaN,NaN,NaN,False
96,2024-04-06,102.961203,NaN,NaN,NaN,False
97,2024-04-07,102.610553,NaN,NaN,NaN,False
98,2024-04-08,100.051135,NaN,NaN,NaN,False


,date,value,rolling mean,rolling_std,z-score,anomaly
20,2024-01-21,150.0,98.865613,13.071006,3.912047,True
45,2024-02-15,50.0,97.361657,12.862075,-3.682272,True
70,2024-03-11,160.0,102.129634,14.859963,3.894381,True


In [18]:
#Ques 5: Compare vectorized groupby cumsum vs apply. Demonstrate memory savings via dtype optimisation..

import time 
np.random.seed(42) 
n = 100_000 
df_l = pd.DataFrame({ "category": np.random.choice(["A","B","C","D"], n), "value": np.random.randint(1, 100, n) }) 
# FAST — vectorized 
start = time.time() 
df_l["running_total"] = df_l.groupby("category")["value"].cumsum()
print(f"Vectorized: {time.time()-start:.4f}s") 
before_mb = df_l.memory_usage(deep=True).sum() / 1e6 
print(f"Memory before: {before_mb:.2f} MB") 
# Optimise dtypes 
df_l["category"] = df_l["category"].astype("category") 
df_l["value"] = df_l["value"].astype("int16") 
after_mb = df_l.memory_usage(deep=True).sum() / 1e6 
print(f"Memory after: {after_mb:.2f} MB") 
print(f"Reduction: {(1-after_mb/before_mb)*100:.0f}%")

Vectorized: 0.0146s
Memory before: 6.60 MB
Memory after: 1.10 MB
Reduction: 83%
